# 05 · Prepare Plates (detector + OCR)

> **OWNER:** Member B (M4 · hazards + plates)
> **PREREQUISITES:** `00_setup_verify.ipynb` all-green. Independent of 03/04 —
> can run before or after them.
> **EXPECTED RUNTIME:** ~1 hour, mostly Step 1's pretrained-model search.
> **OUTPUTS:** either a downloaded+benchmarked pretrained plate detector
> (**and 06 is skipped entirely**), or `data/plates_prepared/data.yaml` ready
> for `06_train_plates.ipynb`; either way, a PaddleOCR smoke test with the
> Indian plate regex validator.

**Next notebook:** `06_train_plates.ipynb` **only if Step 1 doesn't find a
usable pretrained detector** — otherwise skip straight to `07_evaluate_all.ipynb`.

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Step 1 — Check for a pretrained Indian plate detector FIRST

Several Indian number-plate detectors exist on Roboflow Universe and Hugging
Face — a single class, well-represented problem that many people have already
solved. **Check before spending an hour training your own.**

1. Search Roboflow Universe for "indian number plate detection" / "license
   plate India" — several trained models are downloadable directly (not just
   datasets), check each one's licence.
2. Search Hugging Face Hub for the same terms; some are packaged as
   ultralytics-compatible `.pt` checkpoints.
3. If you find one: download it below, benchmark it on your own dashcam
   frames in Step 1b, and **record the decision** — the next cell writes a
   decision file that `06_train_plates.ipynb` reads and skips itself if this
   says "pretrained."

In [ ]:
PLATES_ROOT = DATA_ROOT / "plates"
PLATES_ROOT.mkdir(parents=True, exist_ok=True)

# Fill this in ONLY if Step 1's search found a usable pretrained checkpoint —
# a local path (already downloaded) or a Hugging Face Hub repo id.
PRETRAINED_PLATE_SOURCE = None  # e.g. "/content/drive/MyDrive/downloaded_plate_model.pt" or "some-org/indian-anpr-yolo"

if PRETRAINED_PLATE_SOURCE:
    from pathlib import Path

    if str(PRETRAINED_PLATE_SOURCE).endswith(".pt") and Path(PRETRAINED_PLATE_SOURCE).exists():
        pretrained_path = Path(PRETRAINED_PLATE_SOURCE)
    else:
        from huggingface_hub import hf_hub_download

        pretrained_path = Path(hf_hub_download(repo_id=PRETRAINED_PLATE_SOURCE, filename="best.pt"))
    print(f"using pretrained plate detector: {pretrained_path}")
else:
    pretrained_path = None
    print("PRETRAINED_PLATE_SOURCE is not set — set it above if Step 1's search found something usable,")
    print("otherwise continue to Step 2 to prepare a dataset and train from scratch in 06.")

## Step 1b — benchmark the pretrained model on real dashcam frames (only if PRETRAINED_PLATE_SOURCE is set)

In [ ]:
if pretrained_path is not None:
    from ultralytics import YOLO

    sample_frames_dir = REPO_ROOT / "data" / "raw_video"  # or wherever you have real dashcam stills
    sample_images = sorted(sample_frames_dir.rglob("*.jpg"))[:20] if sample_frames_dir.exists() else []
    if not sample_images:
        print(f"no sample frames found under {sample_frames_dir} to benchmark against — "
              "drop a handful of real dashcam stills there and re-run this cell before deciding.")
    else:
        pretrained_plate_model = YOLO(str(pretrained_path))
        results = pretrained_plate_model.predict(source=[str(p) for p in sample_images], conf=0.25, verbose=False)
        n_with_detection = sum(1 for r in results if len(r.boxes) > 0)
        print(f"{n_with_detection}/{len(sample_images)} sample frames got at least one plate detection")
        print("Look at a few results.boxes / results.plot() outputs by hand before trusting this number —")
        print("a high hit rate on frames that happen to contain no plates is not evidence of quality.")

## Step 1c — record the decision

Writes `models/plate_decision.json`, which `06_train_plates.ipynb` reads at
its own Step 0 to decide whether to skip itself. **Report the decision** —
this cell's printout is the record for the deck too.

In [ ]:
import json

decision = {
    "source": "pretrained" if pretrained_path is not None else "train_from_scratch",
    "pretrained_source": str(PRETRAINED_PLATE_SOURCE) if pretrained_path is not None else None,
}
decision_path = MODEL_ROOT / "plate_decision.json"
decision_path.write_text(json.dumps(decision, indent=2))
print(f"decision written to {decision_path}: {decision}")

if pretrained_path is not None:
    import shutil

    from common import constants

    shutil.copy2(pretrained_path, MODEL_ROOT / constants.MODEL_FILES["plate"])
    print(f"copied pretrained checkpoint -> {MODEL_ROOT / constants.MODEL_FILES['plate']}")
    print()
    print("DECISION: using a pretrained plate detector. 06_train_plates.ipynb will skip itself.")
    print("Continue to Step 2 below anyway for the PaddleOCR smoke test — that part still applies.")
else:
    print()
    print("DECISION: no usable pretrained model found — continue to Step 2 to prepare a training set for 06.")

## Step 2 — Prepare a training dataset (skip if you copied a pretrained checkpoint above and don't intend to fine-tune it)

Roboflow Universe Indian number-plate datasets, single class `PLATE`, index 0
— same import/remap/split shape as notebook 03, just one class.

In [ ]:
import zipfile

from common import constants

export_zip = PLATES_ROOT / "roboflow_export.zip"
export_dir = PLATES_ROOT / "roboflow_export"

if export_zip.exists() and not export_dir.exists():
    with zipfile.ZipFile(export_zip) as zf:
        zf.extractall(export_dir)
    print(f"extracted {export_zip} -> {export_dir}")

if not export_dir.exists():
    print(f"no Roboflow export found at {export_zip} or {export_dir} yet.")
    print("Search Roboflow Universe for an Indian number-plate detection dataset,")
    print("export in YOLO format, and save the zip there before running the rest of this section.")
else:
    import yaml

    roboflow_yaml_path = next(export_dir.rglob("data.yaml"))
    with open(roboflow_yaml_path) as f:
        roboflow_yaml = yaml.safe_load(f)
    print(f"Roboflow export classes: {roboflow_yaml['names']}")
    print(f"expected (frozen): {constants.PLATE_CLASSES}")

In [ ]:
if export_dir.exists():
    from common import splits

    roboflow_images_dir = next(export_dir.rglob("images"))
    roboflow_labels_dir = next(export_dir.rglob("labels"))

    image_class_map = {}
    for label_path in roboflow_labels_dir.rglob("*.txt"):
        classes = {0 for line in label_path.read_text().splitlines() if line.strip()}  # single class -> always {0} if non-empty
        image_class_map[label_path.stem] = classes

    data_splits = splits.stratified_split(image_class_map, ratios=(0.8, 0.1, 0.1), seed=42)
    splits.report_split_balance(image_class_map, data_splits, constants.PLATE_CLASSES)

    OUTPUT_ROOT = DATA_ROOT / "plates_prepared"
    splits.materialize_split(image_class_map, data_splits, roboflow_images_dir, roboflow_labels_dir, OUTPUT_ROOT)

    data_yaml = {"path": str(OUTPUT_ROOT), "train": "images/train", "val": "images/val", "test": "images/test", "names": constants.PLATE_CLASSES}
    constants.assert_class_order(data_yaml["names"], constants.PLATE_CLASSES, "plate")
    (OUTPUT_ROOT / "data.yaml").write_text(yaml.safe_dump(data_yaml, sort_keys=False))
    print(f"wrote {OUTPUT_ROOT / 'data.yaml'}")

    from common import contact_sheet

    contact_sheet.render_contact_sheet(
        images_dir=OUTPUT_ROOT / "images" / "train",
        labels_dir=OUTPUT_ROOT / "labels" / "train",
        class_names=constants.PLATE_CLASSES,
        output_path=OUTPUT_ROOT / "contact_sheet.png",
        n=12,
    )

## Step 3 — PaddleOCR smoke test + Indian plate regex validator

No training here — PaddleOCR is used off-the-shelf. Test it on whatever
cropped plate images you have (from the plate dataset above, or hand-cropped
from dashcam stills), and validate the OCR string against the Indian plate
format. **Expect this to be poor** — mono-camera plate OCR at speed on Indian
roads is genuinely hard. Knowing the real number beats guessing at it, which
is the honest bar to report against, not a target accuracy number.

In [ ]:
import re

INDIAN_PLATE_REGEX = re.compile(r"^[A-Z]{2}[0-9]{1,2}[A-Z]{1,3}[0-9]{4}$")


def validate_indian_plate(text: str) -> bool:
    normalised = "".join(text.split()).upper()
    return bool(INDIAN_PLATE_REGEX.match(normalised))


from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang="en")

# Point this at a folder of cropped plate images — from the plate dataset's
# test split if Step 2 ran, or hand-cropped dashcam stills otherwise.
CROP_TEST_DIR = DATA_ROOT / "plates_prepared" / "images" / "test" if (DATA_ROOT / "plates_prepared").exists() else PLATES_ROOT / "manual_crops"

crop_paths = sorted(CROP_TEST_DIR.glob("*.jpg"))[:20] if CROP_TEST_DIR.exists() else []
if not crop_paths:
    print(f"no crops found at {CROP_TEST_DIR} — drop a handful of cropped plate images there and re-run.")
    print("(Step 2's test-split images are full frames, not crops — for a real OCR test, crop to just the plate region.)")
else:
    n_valid, n_total = 0, 0
    for crop_path in crop_paths:
        result = ocr.ocr(str(crop_path), cls=True)
        texts = [line[1][0] for page in (result or []) for line in (page or [])]
        best_text = max(texts, key=len) if texts else ""
        is_valid = validate_indian_plate(best_text)
        n_total += 1
        n_valid += is_valid
        print(f"{crop_path.name}: OCR='{best_text}'  valid_format={is_valid}")
    print()
    print(f"{n_valid}/{n_total} OCR outputs matched the Indian plate regex format — this is a FORMAT-validity")
    print("rate, not a correctness rate (a well-formed but wrong plate still counts here). Report both numbers")
    print("if you can compare against a small hand-checked ground truth.")

---
### What this notebook produced
- Either: a pretrained plate detector copied to `models/yolo_plate.pt` +
  `models/plate_decision.json` recording the decision (06 is skipped)
- Or: `data/plates_prepared/data.yaml` ready for `06_train_plates.ipynb`
- A PaddleOCR smoke test with the Indian plate regex validator, and an honest
  format-validity number

### Next
`06_train_plates.ipynb` **only if** `plate_decision.json` says `train_from_scratch`.
Otherwise `07_evaluate_all.ipynb`.